# Unduh dan pecah file output apa pun

Untuk Excel, ZIP, CSV, checkpoint, atau file lain dari eksperimen apa pun. Tidak memerlukan kode training.

**Sesi baru:** pilih MODE = "upload", gunakan runtime CPU, lalu Run all dan pilih file dari komputer.
**File masih di runtime training:** salin cell kode notebook ini ke bagian bawah notebook training, pilih MODE = "path", dan isi FILE_PATHS dengan lokasi file output. Runtime Colab yang berbeda tidak berbagi file lokal.
**File di Drive:** pilih MODE = "drive", isi FILE_PATHS dengan path /content/drive/MyDrive/..., lalu Run all.

Ukuran bagian bawaan 30 MiB. Tombol unduh file asli, setiap bagian, dan manifest tersedia di akhir. Untuk dikirim ke chat, unggah seluruh bagian beserta manifest. Bagian biner perlu digabung kembali sebelum dibuka. File diproses di runtime Colab dan tidak dipublikasikan ke GitHub.


## 1 · Pilih sumber file

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from google.colab import files
from IPython.display import display
import ipywidgets as widgets

MODE = "upload"  # "upload", "path", atau "drive"
FILE_PATHS = []  # Contoh: ["/content/hasil.xlsx"]
PART_SIZE_MIB = 30  # Boleh 1 sampai 30 MiB.

if MODE == "upload":
    uploaded = files.upload()
    source_folder = Path(mkdtemp(prefix="file_asli_"))
    source_files = []
    for name, content in uploaded.items():
        path = source_folder / Path(name).name
        path.write_bytes(content)
        source_files.append(path)
    del uploaded
elif MODE in ("path", "drive"):
    if MODE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
    source_files = list(dict.fromkeys(Path(p).expanduser() for p in FILE_PATHS))
else:
    raise ValueError("MODE harus upload, path, atau drive.")

if not source_files:
    raise ValueError("Pilih file atau isi FILE_PATHS terlebih dahulu.")
if not isinstance(PART_SIZE_MIB, int) or not 1 <= PART_SIZE_MIB <= 30:
    raise ValueError("PART_SIZE_MIB harus bilangan bulat 1 sampai 30.")
for path in source_files:
    if not path.is_file() or not path.stat().st_size:
        raise ValueError(f"File tidak ditemukan atau kosong: {path}")
    print(f"{path.name} — {path.stat().st_size / 1024**2:.2f} MiB")


## 2 · Pecah dan verifikasi file

In [ ]:
"""Split an output file into verified pieces suitable for attachment upload."""

import hashlib
import json
from pathlib import Path


def split_output_file(source, destination, part_size=30 * 1024 * 1024):
    """Return a manifest after checking that concatenated pieces match the source."""
    source = Path(source)
    destination = Path(destination)
    if not source.is_file() or source.stat().st_size == 0:
        raise ValueError("Pilih file yang ada dan berisi data.")
    if not 1 <= part_size <= 30 * 1024 * 1024:
        raise ValueError("Ukuran bagian harus antara 1 byte dan 30 MiB.")
    destination.mkdir(parents=True, exist_ok=True)
    source_hash = hashlib.sha256()
    parts = []

    with source.open("rb") as stream:
        number = 1
        while chunk := stream.read(part_size):
            name = f"{source.name}.part{number:03d}"
            path = destination / name
            with path.open("xb") as target:
                target.write(chunk)
            source_hash.update(chunk)
            parts.append({
                "name": name,
                "size_bytes": len(chunk),
                "sha256": hashlib.sha256(chunk).hexdigest(),
            })
            number += 1

    reconstructed_hash = hashlib.sha256()
    total_bytes = 0
    for part in parts:
        part_hash = hashlib.sha256()
        size = 0
        with (destination / part["name"]).open("rb") as stream:
            while block := stream.read(1024 * 1024):
                reconstructed_hash.update(block)
                part_hash.update(block)
                size += len(block)
        if size != part["size_bytes"] or part_hash.hexdigest() != part["sha256"]:
            raise IOError("Verifikasi bagian gagal: " + part["name"])
        total_bytes += size

    if total_bytes != source.stat().st_size or reconstructed_hash.hexdigest() != source_hash.hexdigest():
        raise IOError("Gabungan pecahan berbeda dari file asli.")

    manifest = {
        "original_name": source.name,
        "size_bytes": total_bytes,
        "sha256": source_hash.hexdigest(),
        "part_size_bytes": part_size,
        "parts": parts,
    }
    manifest_path = destination / f"{source.name}.manifest.json"
    with manifest_path.open("x", encoding="utf-8") as target:
        json.dump(manifest, target, ensure_ascii=False, indent=2)
    return manifest, manifest_path


In [ ]:
result_files = []
for source in source_files:
    destination = Path(mkdtemp(prefix="pecahan_output_"))
    manifest, manifest_path = split_output_file(source, destination, PART_SIZE_MIB * 1024 * 1024)
    print(f"\n{source.name}: {len(manifest['parts'])} bagian, verifikasi berhasil.")
    for part in manifest["parts"]:
        result_files.append(destination / part["name"])
        print(f"  {part['name']} — {part['size_bytes'] / 1024**2:.2f} MiB")
    result_files.append(manifest_path)
    print("SHA-256 asli:", manifest["sha256"])


## 3 · Unduh file asli atau pecahannya

Klik satu tombol, tunggu unduhan selesai, lalu klik tombol berikutnya. Untuk mengirim file besar ke chat, unduh semua bagian dan manifest.

In [ ]:
def make_download_button(path):
    button = widgets.Button(
        description="Unduh " + path.name,
        layout=widgets.Layout(width="100%"),
    )
    def download_clicked(_):
        files.download(str(path))
    button.on_click(download_clicked)
    return button

print("File asli:")
display(widgets.VBox([make_download_button(path) for path in source_files]))
print("Bagian dan manifest:")
display(widgets.VBox([make_download_button(path) for path in result_files]))
print("Unggah semua bagian beserta manifest ke chat untuk digabung dan dianalisis.")
